In [1]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from pathlib import Path

/home/ioannis/dev_venv/dlenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load model
model_name = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 22449.94it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [55]:
source_path = Path.cwd().parent
img = Image.open(source_path / "data" / "dog.jpeg").convert("RGB")
text = ["a realistic photo of a dog face"]

In [ ]:
# Load image and texts
source_path = Path.cwd().parent
img = Image.open(source_path / "data" / "dog.jpeg").convert("RGB")
texts = ["a realistic photo of a dog outdoors", "a cat in the amazon"]

# Process inputs separately
image_inputs = processor(images=img, return_tensors="pt")
text_inputs = processor(text=texts, return_tensors="pt", padding=True)

# Move to device
image_inputs = {k: v.to(device) for k, v in image_inputs.items()}
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

# Extract embeddings — TWO safe options depending on what your model returns:

with torch.no_grad():
    image_embedding = model.get_image_features(**image_inputs)
    text_embedding = model.get_text_features(**text_inputs)

# Normalize (required for cosine similarity)
image_embedding = image_embedding / image_embedding.norm(dim=-1, keepdim=True)
text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)

# Cosine similarity scores
similarity = (image_embedding @ text_embedding.T) * 100
print("\nSimilarity scores:")
for i, text in enumerate(texts):
    print(f"  '{text}': {similarity[0, i].item():.2f}")

IMAGE TYPE: <class 'torch.Tensor'>
TEXT TYPE: <class 'torch.Tensor'>
IMAGE SHAPE: torch.Size([1, 512])
TEXT SHAPE: torch.Size([2, 512])

Similarity scores:
  'a realistic photo of a dog outdoors': 25.18
  'a cat in the amazon': 16.08
